# 02 - Image Preprocessing Pipeline

Milestone 2: Preparing the Rakuten product images for reproducible image and multimodal modelling.

This notebook converts the image-audit findings into a reusable preprocessing workflow. It does not train an image model. Raw files remain unchanged, and images are opened only when they are needed.


## 1. Scope and Audit Context

The data-audit notebooks established the following points:

- all 84,916 labelled training products have a matching image file;
- all 2,000 sampled images were readable, RGB images with a size of 500 × 500 pixels;
- exact decoded-image duplicates and train-test overlap were audited for the complete image catalogue;
- the duplicate hashes were cached under `data/processed/`.

The purpose of this notebook is therefore not to repeat the image audit. It prepares an image manifest, reuses the existing audit metadata, loads the validation split created by `02_preprocessing.ipynb`, and defines a deterministic Pillow/NumPy preprocessing function.


## 2. Imports and Configuration

Project paths and image filenames remain centralised in `src.data_utils`. No dataset path is duplicated in this notebook.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

from src.data_utils import (
    get_data_paths,
    get_image_path,
    load_image,
    load_merged_data,
)

RANDOM_STATE = 42
IMAGE_SIZE = (224, 224)

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Environment and data utilities loaded successfully.")


## 3. Build the Training Image Manifest

`load_merged_data()` returns the labelled challenge data. The original dataframe index is retained as `row_id`; this is the safest key for joining text predictions, image predictions and split assignments later in the project.

The manifest stores metadata and file paths only. It does not load 84,916 images into memory.


In [ ]:
data_paths = get_data_paths()

train_df = (
    load_merged_data()
    .rename_axis("row_id")
    .reset_index()
)

train_images = train_df[
    ["row_id", "imageid", "productid", "prdtypecode"]
].copy()
train_images["source_split"] = "train"
train_images["prdtypecode"] = train_images["prdtypecode"].astype("Int64")

# A list comprehension is faster than a row-wise DataFrame.apply for this task.
train_images["image_path"] = [
    Path(
        get_image_path(
            int(row.imageid),
            int(row.productid),
            split="image_train",
        )
    )
    for row in train_images.itertuples(index=False)
]

print("Training image rows:", f"{len(train_images):,}")
display(train_images.head())


## 4. Validate the Manifest

The earlier audit already checked the complete training filename coverage. These assertions protect the preprocessing workflow against accidental path, join or data changes.


In [ ]:
assert train_images["row_id"].is_unique
assert train_images["imageid"].notna().all()
assert train_images["productid"].notna().all()
assert train_images["prdtypecode"].notna().all()

train_images["image_exists"] = train_images["image_path"].map(Path.exists)
available_images = int(train_images["image_exists"].sum())
missing_images = int((~train_images["image_exists"]).sum())

print(f"Available training images: {available_images:,}/{len(train_images):,}")
print(f"Missing training images: {missing_images:,}")

assert train_images["image_exists"].all(), "At least one expected training image is missing."


## 5. Add the Official Test Image Manifest

The official test data has no `prdtypecode` values and must not be used as validation data. It is included here so that exactly the same image-path and preprocessing logic can be checked for both official data sources.

`source_split` means the official challenge source (`train` or `test`). A later `split` column will represent the internal training/validation assignment.


In [ ]:
test_df = (
    pd.read_csv(data_paths["x_test"], index_col=0)
    .rename_axis("row_id")
    .reset_index()
)

test_images = test_df[["row_id", "imageid", "productid"]].copy()
test_images["prdtypecode"] = pd.Series(
    pd.NA,
    index=test_images.index,
    dtype="Int64",
)
test_images["source_split"] = "test"
test_images["image_path"] = [
    Path(
        get_image_path(
            int(row.imageid),
            int(row.productid),
            split="image_test",
        )
    )
    for row in test_images.itertuples(index=False)
]
test_images["image_exists"] = test_images["image_path"].map(Path.exists)

image_catalog = pd.concat(
    [train_images, test_images],
    ignore_index=True,
)

assert not image_catalog.duplicated(["source_split", "row_id"]).any()

catalog_summary = image_catalog.groupby("source_split").agg(
    rows=("row_id", "size"),
    available_images=("image_exists", "sum"),
    labelled_rows=("prdtypecode", "count"),
)
display(catalog_summary)


## 6. Reuse the Existing Duplicate-Image Audit

`01_dataviz_and_audit-BQ.ipynb` created a versioned JSON cache containing exact file and decoded-pixel hashes. Reusing it avoids rescanning every image. The pixel hash is attached to the manifest so identical images can later be kept in the same validation fold.

This cache detects exact decoded-image matches. It does not identify merely similar crops, rescaled files or recompressed JPEGs.


In [ ]:
hash_cache_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "image_duplicate_hashes.json"
)

if hash_cache_path.exists():
    cache_payload = json.loads(hash_cache_path.read_text(encoding="utf-8"))
    hash_records = cache_payload.get("images", {})

    image_catalog["pixel_sha256"] = image_catalog["image_path"].astype(str).map(
        lambda path: hash_records.get(path, {}).get("pixel_sha256")
    )
    image_catalog["image_audit_error"] = image_catalog["image_path"].astype(str).map(
        lambda path: hash_records.get(path, {}).get("error")
    )

    hashed_rows = int(image_catalog["pixel_sha256"].notna().sum())
    print(f"Rows matched to the existing pixel-hash cache: {hashed_rows:,}/{len(image_catalog):,}")
else:
    image_catalog["pixel_sha256"] = pd.NA
    image_catalog["image_audit_error"] = pd.NA
    print("Image hash cache not found. Run the complete image audit before creating duplicate-aware folds.")


## 7. Load the Shared Training/Validation Split

The official test data has no labels, so local evaluation uses the held-out part of the labelled catalogue defined in `02_preprocessing.ipynb`. Loading the same manifest here ensures that text and image classifiers can be evaluated and combined on the same products.

This modality-specific notebook does not create or modify the assignment. Run `02_preprocessing.ipynb` first if `data/processed/data_split.csv` is missing. The baseline split is stratified but not duplicate-aware; a grouped alternative can later be compared as a leakage-sensitivity analysis.


In [ ]:
split_path = (
    Path(data_paths["x_train"]).parent.parent
    / "processed"
    / "data_split.csv"
)

if not split_path.exists():
    raise FileNotFoundError(
        "Shared split manifest not found. "
        "Run notebooks/02_preprocessing.ipynb before this notebook."
    )

split_manifest = pd.read_csv(split_path, usecols=["row_id", "split"])

# Reject stale, incomplete or ambiguous split files before using them.
assert split_manifest["row_id"].is_unique
assert split_manifest["split"].notna().all()
assert set(split_manifest["split"]) == {"train", "validation"}
assert set(split_manifest["row_id"]) == set(train_images["row_id"])

train_images = train_images.drop(columns="split", errors="ignore").merge(
    split_manifest,
    on="row_id",
    how="left",
    validate="one_to_one",
)
assert train_images["split"].notna().all()

split_summary = (
    train_images.groupby("split")
    .agg(
        products=("row_id", "size"),
        classes=("prdtypecode", "nunique"),
    )
)
split_summary["share"] = split_summary["products"] / len(train_images)

class_shares = pd.crosstab(
    train_images["prdtypecode"],
    train_images["split"],
    normalize="columns",
)
max_class_share_difference = (
    class_shares["train"] - class_shares["validation"]
).abs().max()

print(f"Loaded shared split manifest: {split_path}")
display(split_summary.style.format({"share": "{:.1%}"}))
print(
    "Maximum absolute class-share difference between train and validation: "
    f"{max_class_share_difference:.3%}"
)


## 8. Deterministic Image Preprocessing

The audit found square 500 × 500 RGB images in the readability sample. A 224 × 224 output is a practical default for many pretrained convolutional networks. `ImageOps.fit` also keeps the function robust if an image with another aspect ratio is encountered.

The preprocessing steps are:

1. open the image through `src.data_utils`;
2. convert the image to RGB;
3. resize and centre-crop to the requested size;
4. convert pixels to floating-point values in `[0, 1]`;
5. optionally apply ImageNet normalisation.

Validation and test preprocessing must remain deterministic. Random augmentation, if used later, belongs to the training pipeline only and must be justified by validation results.


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def prepare_pil_image(
    imageid: int,
    productid: int,
    source_split: str = "train",
    image_size: tuple[int, int] = IMAGE_SIZE,
) -> Image.Image:
    """Load one product image and return a detached, resized RGB image."""
    split_key = f"image_{source_split}"

    with load_image(
        int(imageid),
        int(productid),
        split=split_key,
    ) as image:
        rgb_image = image.convert("RGB")
        prepared = ImageOps.fit(
            rgb_image,
            image_size,
            method=Image.Resampling.BICUBIC,
        )
        # Copy the result before the source file is closed by the context manager.
        return prepared.copy()


def image_to_array(
    image: Image.Image,
    normalize_imagenet: bool = False,
) -> np.ndarray:
    """Convert a prepared RGB image to a float32 HWC NumPy array."""
    array = np.asarray(image, dtype=np.float32) / 255.0

    if normalize_imagenet:
        array = (array - IMAGENET_MEAN) / IMAGENET_STD

    return array


## 9. Check the Preprocessing Output

A small deterministic sample verifies output size, colour channels, numeric type and finite values. These checks are intentionally lightweight and do not repeat the full readability audit.


In [ ]:
example_row = train_images.sample(1, random_state=RANDOM_STATE).iloc[0]
example_image = prepare_pil_image(
    imageid=example_row["imageid"],
    productid=example_row["productid"],
    source_split=example_row["source_split"],
)
example_array = image_to_array(example_image, normalize_imagenet=True)

assert example_image.mode == "RGB"
assert example_image.size == IMAGE_SIZE
assert example_array.shape == (IMAGE_SIZE[1], IMAGE_SIZE[0], 3)
assert example_array.dtype == np.float32
assert np.isfinite(example_array).all()

print("Prepared PIL image size:", example_image.size)
print("Prepared array shape:", example_array.shape)
print("Prepared array dtype:", example_array.dtype)
print("All values are finite:", bool(np.isfinite(example_array).all()))


## 10. Visual Check of Prepared Images

The gallery below displays the deterministic model inputs before numerical normalisation. It helps detect incorrect orientation, unexpected cropping or broken colour conversion.


In [ ]:
preview_rows = train_images.sample(8, random_state=RANDOM_STATE)
fig, axes = plt.subplots(2, 4, figsize=(12, 7))

for axis, row in zip(axes.flat, preview_rows.itertuples(index=False)):
    prepared_image = prepare_pil_image(
        imageid=row.imageid,
        productid=row.productid,
        source_split=row.source_split,
    )
    axis.imshow(prepared_image)
    axis.set_title(f"Class {int(row.prdtypecode)} | row {row.row_id}", fontsize=9)
    axis.axis("off")

fig.suptitle("Deterministically prepared 224 × 224 training images", fontweight="bold")
plt.tight_layout()
plt.show()


## 11. Framework Integration Decision

The current project dependencies include Pillow but do not yet include PyTorch, torchvision or TensorFlow. This notebook therefore stops at a framework-neutral Pillow/NumPy representation.

Before image modelling begins, the team should select one framework and one pretrained backbone. The final training loader should then:

- reuse the same manifest and `row_id`;
- load images lazily rather than materialising all transformed files;
- use the preprocessing supplied by the selected pretrained weights when available;
- apply random augmentation to training rows only;
- keep validation and test transformations deterministic;
- return the target class only for labelled training rows.

If ImageNet-pretrained weights are selected, the mean and standard deviation demonstrated above are the usual normalisation values. They should still be checked against the exact weights API instead of being assumed for every model.


## 12. Preprocessing Decisions and Next Step

| Audit finding | Preprocessing decision |
|---|---|
| Complete training filename coverage | Keep every labelled row and retain a missing-file assertion |
| Sampled images are readable, square and RGB | Convert robustly to RGB and resize to the model input size |
| Exact duplicate images exist | Attach the cached pixel hash and compare the baseline split with a duplicate-aware grouped split later |
| Test labels are unavailable | Keep official test rows separate from internal validation |
| Text and image predictions will later be combined | Preserve `row_id` as the shared one-to-one join key |
| No image modelling framework is currently installed | Keep preprocessing framework-neutral until a backbone is selected |

The shared stratified split is loaded from the general preprocessing notebook. The next implementation step is to wrap the lazy preprocessing function in the dataset interface of the selected image framework. Model training belongs to the modelling milestone rather than this preprocessing notebook.
